## Phase 5 — Embeddings + Vector Search Index
**Reads from:** `main.silver.news_articles`
**Creates:**
- `main.silver.news_for_search` — CDF-enabled source table
- Vector Search endpoint: `stock-assistant-vs`
- Vector Search index: `main.silver.news_for_search_index`

Uses Databricks Managed Embeddings (BGE Large).
Run Cell 1 first — it installs the SDK and restarts Python.
Then Run All again from Cell 2.


In [ ]:
# 1. Install Vector Search SDK
# Kernel restarts after this — Run All again after restart
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()


In [ ]:
# 2. Imports and config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from databricks.vector_search.client import VectorSearchClient
from datetime import datetime
import time

spark = SparkSession.builder.getOrCreate()

PROCESSED_AT    = datetime.now().isoformat()
ENDPOINT_NAME   = "stock-assistant-vs"
INDEX_NAME      = "main.silver.news_for_search_index"
SOURCE_TABLE    = "main.silver.news_for_search"
EMBEDDING_MODEL = "databricks-bge-large-en"

print(f"Started  : {PROCESSED_AT}")
print(f"Endpoint : {ENDPOINT_NAME}")
print(f"Index    : {INDEX_NAME}")
print(f"Model    : {EMBEDDING_MODEL}")


In [ ]:
# 3. Prepare source table for Vector Search
# Spec requires embedding: news articles + company profiles
# We union both to enable semantic queries like:
# "companies exposed to AI chip demand" or "pharmaceutical companies with strong pipeline"
print("\n--- Preparing source table (news + company profiles) ---")

# ── Part A: News articles ─────────────────────────────────────────────────────
news_for_search = (
    spark.table("main.silver.news_articles")
    .withColumn("search_text",
        F.concat_ws(" | ",
            F.col("ticker"),
            F.col("title"),
            F.coalesce(F.col("description"), F.lit(""))
        )
    )
    .select(
        "article_id", "ticker", "title", "description",
        "search_text", "publisher_name", "sentiment",
        "published_utc", "published_ts",
        "article_age_days", "has_sentiment", "processed_at"
    )
    .filter(F.col("article_id").isNotNull())
    .filter(F.col("search_text").isNotNull())
)

news_count = news_for_search.count()
print(f"News articles: {news_count} rows")

# ── Part B: Company profiles ──────────────────────────────────────────────────
# Embed company name + business description so agent can find
# "semiconductor companies" or "regional banks" semantically
companies_for_search = (
    spark.table("main.silver.companies")
    .filter(F.col("description").isNotNull())
    .withColumn("search_text",
        F.concat_ws(" | ",
            F.col("ticker"),
            F.col("name"),
            F.coalesce(F.col("description"), F.lit(""))
        )
    )
    .withColumn("article_id",
        F.concat(F.lit("company_"), F.col("ticker"))   # unique ID, no clash with news
    )
    .withColumn("title",          F.col("name"))
    .withColumn("publisher_name", F.lit("Company Profile"))
    .withColumn("sentiment",      F.lit("neutral"))
    .withColumn("published_utc",  F.lit("2026-08-05"))
    .withColumn("published_ts",   F.to_timestamp(F.lit("2026-08-05")))
    .withColumn("article_age_days", F.lit(0))
    .withColumn("has_sentiment",  F.lit(False))
    .withColumn("processed_at",   F.col("processed_at"))
    .select(
        "article_id", "ticker", "title", "description",
        "search_text", "publisher_name", "sentiment",
        "published_utc", "published_ts",
        "article_age_days", "has_sentiment", "processed_at"
    )
    .filter(F.col("search_text").isNotNull())
)

company_count = companies_for_search.count()
print(f"Company profiles: {company_count} rows")

# ── Union both sources ────────────────────────────────────────────────────────
combined = news_for_search.union(companies_for_search)
total = combined.count()
print(f"Total for embedding: {total} rows (news + company profiles)")

(combined
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SOURCE_TABLE))

# Enable CDF — required for Delta Sync index auto-updates
spark.sql(f"""
    ALTER TABLE {SOURCE_TABLE}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

count = spark.table(SOURCE_TABLE).count()
print(f"Written {count} rows → {SOURCE_TABLE} ✓")
print("CDF enabled ✓")

print("\nSample search_text:")
spark.table(SOURCE_TABLE) \
     .select("article_id", "ticker", "search_text") \
     .show(5, truncate=90)


In [ ]:
# 4. Create Vector Search endpoint (or confirm existing)
print("\n--- Setting up Vector Search endpoint ---")

vsc = VectorSearchClient(disable_notice=True)

try:
    ep    = vsc.get_endpoint(ENDPOINT_NAME)
    state = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
    print(f"Endpoint '{ENDPOINT_NAME}' already exists — state: {state}")
except Exception:
    print(f"Creating endpoint '{ENDPOINT_NAME}'...")
    vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    print("Endpoint creation started...")

# Wait for ONLINE
print("Waiting for endpoint ONLINE...")
for i in range(20):
    try:
        state = vsc.get_endpoint(ENDPOINT_NAME) \
                   .get("endpoint_status", {}).get("state", "UNKNOWN")
        print(f"  [{i+1}/20] Endpoint state: {state}")
        if state == "ONLINE":
            break
    except Exception as e:
        print(f"  [{i+1}/20] Waiting... ({e})")
    time.sleep(30)

print(f"Endpoint '{ENDPOINT_NAME}' is ONLINE ✓")


In [ ]:
# 5. Delete stuck index if exists, then recreate
# NOTE: If index was created while endpoint was still provisioning,
# it gets stuck in PROVISIONING_ENDPOINT state forever.
# Safe fix: delete and recreate now that endpoint is stable.
print("\n--- Setting up Vector Search index ---")

vsc = VectorSearchClient(disable_notice=True)

try:
    idx   = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)
    desc  = idx.describe()
    state = desc.get("status", {}).get("detailed_state", "UNKNOWN")
    ready = desc.get("status", {}).get("ready", False)
    print(f"Existing index found — state: {state} | ready: {ready}")

    # If stuck in PROVISIONING_ENDPOINT delete and recreate
    if state == "PROVISIONING_ENDPOINT" and not ready:
        print("Index stuck in PROVISIONING_ENDPOINT — deleting and recreating...")
        vsc.delete_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        print("Deleted ✓ — waiting 60s for cleanup...")
        time.sleep(60)
        raise Exception("Recreate needed")
    else:
        print("Index is in a valid state — skipping creation")

except Exception as e:
    if "Recreate needed" in str(e) or "does not exist" in str(e).lower() or "not found" in str(e).lower():
        print(f"Creating index '{INDEX_NAME}'...")
        vsc.create_delta_sync_index(
            endpoint_name                 = ENDPOINT_NAME,
            index_name                    = INDEX_NAME,
            source_table_name             = SOURCE_TABLE,
            pipeline_type                 = "TRIGGERED",
            primary_key                   = "article_id",
            embedding_source_column       = "search_text",
            embedding_model_endpoint_name = EMBEDDING_MODEL
        )
        print("Index creation started ✓")
    else:
        print(f"Unexpected error: {e}")


In [ ]:
# 6. Wait for index READY
# Provisioning stages: PROVISIONING_ENDPOINT → PROVISIONING_PIPELINE_RESOURCES → ONLINE
# Free Edition can take 20-40 minutes total
print("\n--- Waiting for index ready ---")
print("Provisioning stages:")
print("  1. PROVISIONING_ENDPOINT         (endpoint slice allocation)")
print("  2. PROVISIONING_PIPELINE_RESOURCES (embedding pipeline setup)")
print("  3. ONLINE / ready: True          (search available)")
print()

vsc = VectorSearchClient(disable_notice=True)

for i in range(40):
    try:
        desc  = vsc.get_index(ENDPOINT_NAME, INDEX_NAME).describe()
        state = desc.get("status", {}).get("detailed_state", "UNKNOWN")
        ready = desc.get("status", {}).get("ready", False)
        print(f"  [{i+1}/40] State: {state} | Ready: {ready}")
        if ready:
            print("\nIndex is READY ✓")
            break
    except Exception as e:
        print(f"  [{i+1}/40] Error: {e}")
    time.sleep(30)
else:
    print("\nTimeout — index still provisioning.")
    print("Run the status check cell manually every 5 min:")
    print("  desc = vsc.get_index(ENDPOINT_NAME, INDEX_NAME).describe()")
    print("  print(desc.get('status', {}))")


In [ ]:
# 7. Trigger index sync + test semantic search
print("\n--- Triggering index sync ---")

vsc = VectorSearchClient(disable_notice=True)

try:
    idx = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)
    idx.sync()
    print("Sync triggered ✓ — waiting 90s...")
    time.sleep(90)
except Exception as e:
    print(f"Sync note: {e}")

print("\n--- Testing semantic search ---")

test_queries = [
    "Apple stock price drop earnings",
    "AI artificial intelligence technology growth",
    "market volatility interest rates",
]

try:
    idx = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)
    for query in test_queries:
        print(f"\nQuery: '{query}'")
        results = idx.similarity_search(
            query_text  = query,
            columns     = ["ticker", "title", "sentiment"],
            num_results = 3
        )
        hits = results.get("result", {}).get("data_array", [])
        if hits:
            for i, hit in enumerate(hits, 1):
                print(f"  {i}. [{hit[0]}] {str(hit[1])[:70]} ({hit[2]})")
        else:
            print("  No results — index may still be syncing, retry in 2 min")
except Exception as e:
    print(f"Search note: {e}")


In [ ]:
# 8. Status check cell — run manually anytime to check index state
print("=== Manual Status Check ===")

vsc  = VectorSearchClient(disable_notice=True)
desc = vsc.get_index(ENDPOINT_NAME, INDEX_NAME).describe()

print(f"Endpoint : {ENDPOINT_NAME}")
print(f"Index    : {INDEX_NAME}")
print(f"State    : {desc.get('status', {}).get('detailed_state', 'UNKNOWN')}")
print(f"Ready    : {desc.get('status', {}).get('ready', False)}")
print(f"Message  : {desc.get('status', {}).get('message', '')}")
print(f"URL      : {desc.get('status', {}).get('index_url', '')}")
print(f"Rows     : {spark.table(SOURCE_TABLE).count()}")


In [ ]:
# 9. Summary
print("\n=== Phase 5 Summary ===")
print(f"Source table  : {SOURCE_TABLE}")
print(f"VS Endpoint   : {ENDPOINT_NAME}")
print(f"VS Index      : {INDEX_NAME}")
print(f"Embed model   : {EMBEDDING_MODEL}")
print(f"Rows indexed  : {spark.table(SOURCE_TABLE).count()}")
print("""
Data flow:
  news_for_search (CDF enabled)
       ↓ Delta Sync via Change Data Feed
  Vector Search index (BGE Large managed embeddings)
       ↓ similarity_search(query_text, num_results=3)
  Agent tool: search_news(query)
       ↓ top-k articles returned as RAG context
""")
print("Phase 5 complete ✓")
